# bnr_lab-2

Lets have quick review on 
- Cache
- Bucketing

In [1]:
from pyspark.sql import functions as F

In [2]:
# Setting Storage Partitioned Joins (SPJ) related configs
# these are dynamic settings (not static ones), so we can set them directly in a running SparkSession

spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true') 

spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

spark.conf.set('spark.sql.sources.v2.bucketing.pushPartValues.enabled','true')
# When enabled, try to eliminate shuffle if one side of the join has missing partition values from the other side.

spark.conf.set('spark.sql.requireAllClusterKeysForCoPartition','false')
# When true, require the join or MERGE keys to be same and in the same order as the partition keys to eliminate shuffle.

spark.conf.set('spark.sql.sources.v2.bucketing.partiallyClusteredDistribution.enabled','true')
# When true, and when the join is not a full outer join, 
# enable skew optimizations to handle partitions with large amounts of data when avoiding shuffle.
# With this configuration enabled, Spark breaks the skewed partitions into multiple splits
# and the other side of same partition will be grouped and replicated to match the same (similar to salting).

# only on spark>=4.0
# spark.sql.sources.v2.bucketing.allowJoinKeysSubsetOfPartitionKeys.enabled
# When enabled, try to avoid shuffle even if join condition does not include all partition columns.
# e.g., table A has bucket(2, id), and table B has bucket(4, id)

# for more details see:
# Shuffle-less Join, a.k.a Storage Partition Join in Apache Spark - Why, How and Where?
# https://www.guptaakashdeep.com/storage-partition-join-in-apache-spark-why-how-and-where/

## Cache

In [3]:
!ls /home/iceberg/data

devices.csv  maps.csv		matches.csv  medals_matches_players.csv
events.csv   match_details.csv	medals.csv


In [4]:
# path = '/home/iceberg/data/matches.csv'
# df_matches = spark.read.option("header", "true").csv(path)

In [5]:
path = '/home/iceberg/data/match_details.csv'
df_match_details = spark.read.option("header", "true").csv(path)

In [6]:
print(df_match_details.count())
df_match_details.printSchema()

151761
root
 |-- match_id: string (nullable = true)
 |-- player_gamertag: string (nullable = true)
 |-- previous_spartan_rank: string (nullable = true)
 |-- spartan_rank: string (nullable = true)
 |-- previous_total_xp: string (nullable = true)
 |-- total_xp: string (nullable = true)
 |-- previous_csr_tier: string (nullable = true)
 |-- previous_csr_designation: string (nullable = true)
 |-- previous_csr: string (nullable = true)
 |-- previous_csr_percent_to_next_tier: string (nullable = true)
 |-- previous_csr_rank: string (nullable = true)
 |-- current_csr_tier: string (nullable = true)
 |-- current_csr_designation: string (nullable = true)
 |-- current_csr: string (nullable = true)
 |-- current_csr_percent_to_next_tier: string (nullable = true)
 |-- current_csr_rank: string (nullable = true)
 |-- player_rank_on_team: string (nullable = true)
 |-- player_finished: string (nullable = true)
 |-- player_average_life: string (nullable = true)
 |-- player_total_kills: string (nullable = t

In [7]:
# checking if ("match_id", "player_gamertag") is the composite primary key
# yes. these are the primary keys
(df_match_details
 .groupBy("match_id", "player_gamertag")
 .agg(F.count("*").alias("c"))
 .orderBy(F.col("c").desc())
 .limit(5)
 .show())

[Stage 4:================>                                          (2 + 5) / 7]

+--------------------+---------------+---+
|            match_id|player_gamertag|  c|
+--------------------+---------------+---+
|322ae96f-7c46-450...| Darknight 1993|  1|
|5bf67913-da69-491...| Real Navy SEAL|  1|
|7e45c39a-0210-40f...|     GodsBakery|  1|
|703eea99-410c-499...|    Daddy Sseth|  1|
|f9d35e23-70fa-40f...|ScaryGremlinFTW|  1|
+--------------------+---------------+---+



In [8]:
# for each player, lets get their total kill count over all matches

In [9]:
(df_match_details
 .select("match_id", "player_gamertag", "player_total_kills")
 .where(F.col("player_gamertag") == F.lit("EcZachly")).show(5))

+--------------------+---------------+------------------+
|            match_id|player_gamertag|player_total_kills|
+--------------------+---------------+------------------+
|71d79b23-4143-435...|       EcZachly|                12|
|fc3cefc0-954a-456...|       EcZachly|                 9|
|322ae96f-7c46-450...|       EcZachly|                17|
|ccdc92bf-aa22-494...|       EcZachly|                11|
|155cfd23-4f97-4f1...|       EcZachly|                16|
+--------------------+---------------+------------------+
only showing top 5 rows



In [10]:
df_best_players = (df_match_details
 .groupBy("player_gamertag")
 .agg(
     F.sum("player_total_kills").alias("sum_player_total_kills"),
     F.avg("player_total_kills").alias("avg_player_total_kills"),
     F.count("*").alias("count_matches"),
 )
 .orderBy(
     ["sum_player_total_kills", "avg_player_total_kills", "count_matches"],
     ascending=False
 )
).cache()

In [16]:
df_best_players.show(5)

[Stage 19:========>                                                 (1 + 6) / 7]

+---------------+----------------------+----------------------+-------------+
|player_gamertag|sum_player_total_kills|avg_player_total_kills|count_matches|
+---------------+----------------------+----------------------+-------------+
|       EcZachly|               35537.0|     10.68781954887218|         3325|
|    ILLICIT 117|               17624.0|     9.742399115533443|         1809|
|  JakeWilson801|               12615.0|     9.779069767441861|         1290|
| Ash All Mighty|               11251.0|      8.47852298417483|         1327|
|      Amplafied|               10003.0|    10.145030425963489|          986|
+---------------+----------------------+----------------------+-------------+
only showing top 5 rows



In [12]:
df_match_details.join(df_best_players, "player_gamertag").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [player_gamertag#18, match_id#17, previous_spartan_rank#19, spartan_rank#20, previous_total_xp#21, total_xp#22, previous_csr_tier#23, previous_csr_designation#24, previous_csr#25, previous_csr_percent_to_next_tier#26, previous_csr_rank#27, current_csr_tier#28, current_csr_designation#29, current_csr#30, current_csr_percent_to_next_tier#31, current_csr_rank#32, player_rank_on_team#33, player_finished#34, player_average_life#35, player_total_kills#36, player_total_headshots#37, player_total_weapon_damage#38, player_total_shots_landed#39, player_total_melee_kills#40, ... 15 more fields]
   +- BroadcastHashJoin [player_gamertag#18], [player_gamertag#394], Inner, BuildRight, false
      :- Filter isnotnull(player_gamertag#18)
      :  +- FileScan csv [match_id#17,player_gamertag#18,previous_spartan_rank#19,spartan_rank#20,previous_total_xp#21,total_xp#22,previous_csr_tier#23,previous_csr_designation#24,previous_csr#25,previo

26/05/29 22:55:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [13]:
# note that the cache does not show in the execution plan though.

In [14]:
# If you build the same logical plan again in another Python variable,
# Spark can benefit from the cache because caching is tied to the Spark plan/session, not the Python variable name.

In [15]:
df_best_players.unpersist()

DataFrame[player_gamertag: string, sum_player_total_kills: double, avg_player_total_kills: double, count_matches: bigint]

### Just quick check on our environment 

In [17]:
print(spark.sparkContext._jsc.sc().getExecutorMemoryStatus()) # memory of each executor

Map(b200b2b47fe2:35563 -> (455501414,455380380))


In [18]:
print(spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size()) # number of executors

1


In [19]:
spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() # memory of the drive

1073741824

In [ ]:
# we are running on local mode though, so the driver and executor are even in the same jvm.
# ('spark.master', 'local[*]')
# “one executor using all available cores as task threads”

In [56]:
# spark.sparkContext.getConf().getAll()

In [21]:
# we could not use much memory anyway, this computer has only 8GB of RAM

## Buckets

In [20]:
# avoid broadcast join, this way we can see the bucket join happening
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [21]:
!ls /home/iceberg/data

devices.csv  maps.csv		matches.csv  medals_matches_players.csv
events.csv   match_details.csv	medals.csv


In [22]:
path = "/home/iceberg/data/matches.csv"
df_matches = spark.read.option("header", "true").csv(path)

In [23]:
# path = "/home/iceberg/data/match_details.csv"
# df_match_details = spark.read.option("header", "true").csv(path)

In [24]:
df_matches_t = (
    df_matches
    .withColumn("completion_date", F.to_date(F.col("completion_date")))
    .where((F.col("completion_date")>='2016-01-01') & (F.col("completion_date")<='2016-01-5'))
    .select("match_id", "is_team_game", "playlist_id", "completion_date")
)

In [25]:
df_matches_t.printSchema()

root
 |-- match_id: string (nullable = true)
 |-- is_team_game: string (nullable = true)
 |-- playlist_id: string (nullable = true)
 |-- completion_date: date (nullable = true)



In [67]:
(df_matches_t
 .writeTo("demo.bootcamp.matches_p") # partitioned
 .partitionedBy("completion_date")
 .createOrReplace()
)

In [68]:
(df_matches_t
 .writeTo("demo.bootcamp.matches_b") # bucketed
 .partitionedBy(F.bucket(16, "match_id"))
 .createOrReplace()
)

In [69]:
(df_matches_t
 .writeTo("demo.bootcamp.matches_pb") # partitioned and bucketed
 .partitionedBy("completion_date", F.bucket(16, "match_id"))
 .createOrReplace()
)

In [70]:
(df_matches_t
 .where((F.col("completion_date")>='2016-01-03'))
 .writeTo("demo.bootcamp.matches_pd") # with diff number of partitions
 .partitionedBy("completion_date")
 .createOrReplace()
)

In [71]:
(df_matches_t
 .writeTo("demo.bootcamp.matches_bdm") # with diff number of buckets, but multiple
 .partitionedBy(F.bucket(8, "match_id"))
 .createOrReplace()
)

In [72]:
(df_matches_t
 .writeTo("demo.bootcamp.matches_bd") # with diff number of buckets, not multiple
 .partitionedBy(F.bucket(5, "match_id"))
 .createOrReplace()
)

In [45]:
print(spark.sql("show create table demo.bootcamp.matches_bd").collect()[0][0])

CREATE TABLE demo.bootcamp.matches_bd (
  match_id STRING,
  is_team_game STRING,
  playlist_id STRING,
  completion_date DATE)
USING iceberg
CLUSTERED BY (match_id)
INTO 5 BUCKETS
LOCATION 's3://warehouse/bootcamp/matches_bd'
TBLPROPERTIES (
  'created-at' = '2026-05-29T22:59:22.498623926Z',
  'current-snapshot-id' = '3240953509940588097',
  'format' = 'iceberg/parquet',
  'format-version' = '2',
  'write.parquet.compression-codec' = 'zstd')



In [46]:
t_types = [
    'demo.bootcamp.matches_p', # partitioned
    'demo.bootcamp.matches_b', # bucketed
    'demo.bootcamp.matches_pb', # partitioned and bucketed
    'demo.bootcamp.matches_pd', # with diff number of partitions
    'demo.bootcamp.matches_bdm', # with diff number of buckets, but multiple
    'demo.bootcamp.matches_bd' # with diff number of buckets
]

df_d = {}
for t in t_types:
    df_d[t.split('.')[-1]] = spark.read.table(t)

In [47]:
def run_join_analysis(join_d):
    global join_d_l
    
    df_a = df_d[join_d['a']]
    df_b = df_d[join_d['b']]
    
    j = df_a.join(df_b, on=join_d['on'])
    
    plan = j._jdf.queryExecution().executedPlan().toString()
    join_d['has_shuffle'] = 'Exchange' in plan
    
    join_d_l.append(join_d)
    return j

In [48]:
join_d_l = []

In [49]:
# both have same partitions
join_d = {'a':'matches_p', 'b': 'matches_p', 'on':['completion_date']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1768, match_id#1765, is_team_game#1766, playlist_id#1767, match_id#1813, is_team_game#1814, playlist_id#1815]
   +- SortMergeJoin [completion_date#1768], [completion_date#1816], Inner
      :- Sort [completion_date#1768 ASC NULLS FIRST], false, 0
      :  +- BatchScan demo.bootcamp.matches_p[match_id#1765, is_team_game#1766, playlist_id#1767, completion_date#1768] demo.bootcamp.matches_p (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []
      +- Sort [completion_date#1816 ASC NULLS FIRST], false, 0
         +- BatchScan demo.bootcamp.matches_p[match_id#1813, is_team_game#1814, playlist_id#1815, completion_date#1816] demo.bootcamp.matches_p (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []




In [50]:
# both have same bucket column and number of buckets
join_d = {'a':'matches_b', 'b': 'matches_b', 'on':['match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776, is_team_game#1843, playlist_id#1844, completion_date#1845]
   +- SortMergeJoin [match_id#1773], [match_id#1842], Inner
      :- Sort [match_id#1773 ASC NULLS FIRST], false, 0
      :  +- BatchScan demo.bootcamp.matches_b[match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776] demo.bootcamp.matches_b (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
      +- Sort [match_id#1842 ASC NULLS FIRST], false, 0
         +- BatchScan demo.bootcamp.matches_b[match_id#1842, is_team_game#1843, playlist_id#1844, completion_date#1845] demo.bootcamp.matches_b (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []




In [51]:
# both have same partitions, bucket column and number of buckets
join_d = {'a':'matches_pb', 'b': 'matches_pb', 'on':['completion_date', 'match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1784, match_id#1781, is_team_game#1782, playlist_id#1783, is_team_game#1874, playlist_id#1875]
   +- SortMergeJoin [completion_date#1784, match_id#1781], [completion_date#1876, match_id#1873], Inner
      :- Sort [completion_date#1784 ASC NULLS FIRST, match_id#1781 ASC NULLS FIRST], false, 0
      :  +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=completion_date, match_id_bucket] RuntimeFilters: []
      +- Sort [completion_date#1876 ASC NULLS FIRST, match_id#1873 ASC NULLS FIRST], false, 0
         +- BatchScan demo.bootcamp.matches_pb[match_id#1873, is_team_game#1874, playlist_id#1875, completion_date#1876] demo.bootcamp.matches_pb (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=completion_date,

In [52]:
# both are partitioned by the same column, but they have diff number of partitions
join_d = {'a':'matches_p', 'b': 'matches_pd', 'on':['completion_date']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1768, match_id#1765, is_team_game#1766, playlist_id#1767, match_id#1789, is_team_game#1790, playlist_id#1791]
   +- SortMergeJoin [completion_date#1768], [completion_date#1792], Inner
      :- Sort [completion_date#1768 ASC NULLS FIRST], false, 0
      :  +- BatchScan demo.bootcamp.matches_p[match_id#1765, is_team_game#1766, playlist_id#1767, completion_date#1768] demo.bootcamp.matches_p (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []
      +- Sort [completion_date#1792 ASC NULLS FIRST], false, 0
         +- BatchScan demo.bootcamp.matches_pd[match_id#1789, is_team_game#1790, playlist_id#1791, completion_date#1792] demo.bootcamp.matches_pd (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []




In [53]:
# both are bucketed by the same column, but they have diff number of buckets, though the numbers are multiple
join_d = {'a':'matches_b', 'b': 'matches_bdm', 'on':['match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776, is_team_game#1798, playlist_id#1799, completion_date#1800]
   +- SortMergeJoin [match_id#1773], [match_id#1797], Inner
      :- Sort [match_id#1773 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#1773, 200), ENSURE_REQUIREMENTS, [plan_id=2222]
      :     +- BatchScan demo.bootcamp.matches_b[match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776] demo.bootcamp.matches_b (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
      +- Sort [match_id#1797 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(match_id#1797, 200), ENSURE_REQUIREMENTS, [plan_id=2223]
            +- BatchScan demo.bootcamp.matches_bdm[match_id#1797, is_team_game#1798, playlist_id#1799, completion_date#1800] demo.bootcamp.matches_bdm (branch=null) [filters=match_id IS NOT NULL, gro

In [54]:
# both are bucketed by the same column, but they have diff number of buckets, though the numbers are multiple
join_d = {'a':'matches_b', 'b': 'matches_bd', 'on':['match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776, is_team_game#1806, playlist_id#1807, completion_date#1808]
   +- SortMergeJoin [match_id#1773], [match_id#1805], Inner
      :- Sort [match_id#1773 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#1773, 200), ENSURE_REQUIREMENTS, [plan_id=2245]
      :     +- BatchScan demo.bootcamp.matches_b[match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776] demo.bootcamp.matches_b (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
      +- Sort [match_id#1805 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(match_id#1805, 200), ENSURE_REQUIREMENTS, [plan_id=2246]
            +- BatchScan demo.bootcamp.matches_bd[match_id#1805, is_team_game#1806, playlist_id#1807, completion_date#1808] demo.bootcamp.matches_bd (branch=null) [filters=match_id IS NOT NULL, group

In [55]:
# only the partition in common
join_d = {'a':'matches_p', 'b': 'matches_pb', 'on':['completion_date']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1768, match_id#1765, is_team_game#1766, playlist_id#1767, match_id#1781, is_team_game#1782, playlist_id#1783]
   +- SortMergeJoin [completion_date#1768], [completion_date#1784], Inner
      :- Sort [completion_date#1768 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(completion_date#1768, 200), ENSURE_REQUIREMENTS, [plan_id=2269]
      :     +- BatchScan demo.bootcamp.matches_p[match_id#1765, is_team_game#1766, playlist_id#1767, completion_date#1768] demo.bootcamp.matches_p (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []
      +- Sort [completion_date#1784 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(completion_date#1784, 200), ENSURE_REQUIREMENTS, [plan_id=2268]
            +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=

In [56]:
# only the bucket in common
join_d = {'a':'matches_b', 'b': 'matches_pb', 'on':['match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776, is_team_game#1782, playlist_id#1783, completion_date#1784]
   +- SortMergeJoin [match_id#1773], [match_id#1781], Inner
      :- Sort [match_id#1773 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#1773, 200), ENSURE_REQUIREMENTS, [plan_id=2292]
      :     +- BatchScan demo.bootcamp.matches_b[match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776] demo.bootcamp.matches_b (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
      +- Sort [match_id#1781 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(match_id#1781, 200), ENSURE_REQUIREMENTS, [plan_id=2291]
            +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=null) [filters=match_id IS NOT NULL, group

In [57]:
# the join keys are a superset of the partition
join_d = {'a':'matches_p', 'b': 'matches_p', 'on':['completion_date', 'match_id'], 'note__join_keys': 'superset'}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1768, match_id#1765, is_team_game#1766, playlist_id#1767, is_team_game#2001, playlist_id#2002]
   +- SortMergeJoin [completion_date#1768, match_id#1765], [completion_date#2003, match_id#2000], Inner
      :- Sort [completion_date#1768 ASC NULLS FIRST, match_id#1765 ASC NULLS FIRST], false, 0
      :  +- Filter isnotnull(match_id#1765)
      :     +- BatchScan demo.bootcamp.matches_p[match_id#1765, is_team_game#1766, playlist_id#1767, completion_date#1768] demo.bootcamp.matches_p (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=completion_date] RuntimeFilters: []
      +- Sort [completion_date#2003 ASC NULLS FIRST, match_id#2000 ASC NULLS FIRST], false, 0
         +- Filter isnotnull(match_id#2000)
            +- BatchScan demo.bootcamp.matches_p[match_id#2000, is_team_game#2001, playlist_id#2002, completion_date#2003] demo.bootcamp.matches_p (branch=null) [filters=comp

In [58]:
# the join keys are a superset of the bucket
join_d = {'a':'matches_b', 'b': 'matches_b', 'on':['completion_date', 'match_id'], 'note__join_keys': 'superset'}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1776, match_id#1773, is_team_game#1774, playlist_id#1775, is_team_game#2024, playlist_id#2025]
   +- SortMergeJoin [completion_date#1776, match_id#1773], [completion_date#2026, match_id#2023], Inner
      :- Sort [completion_date#1776 ASC NULLS FIRST, match_id#1773 ASC NULLS FIRST], false, 0
      :  +- Filter isnotnull(completion_date#1776)
      :     +- BatchScan demo.bootcamp.matches_b[match_id#1773, is_team_game#1774, playlist_id#1775, completion_date#1776] demo.bootcamp.matches_b (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
      +- Sort [completion_date#2026 ASC NULLS FIRST, match_id#2023 ASC NULLS FIRST], false, 0
         +- Filter isnotnull(completion_date#2026)
            +- BatchScan demo.bootcamp.matches_b[match_id#2023, is_team_game#2024, playlist_id#2025, completion_date#2026] demo.bootcamp.matches_b (branch=null)

In [59]:
# the join keys are a subset of the partition/bucket
join_d = {'a':'matches_pb', 'b': 'matches_pb', 'on':['match_id'], 'note__join_keys': 'subset'}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784, is_team_game#2049, playlist_id#2050, completion_date#2051]
   +- SortMergeJoin [match_id#1781], [match_id#2048], Inner
      :- Sort [match_id#1781 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(match_id#1781, 200), ENSURE_REQUIREMENTS, [plan_id=2384]
      :     +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=null) [filters=match_id IS NOT NULL, groupedBy=completion_date, match_id_bucket] RuntimeFilters: []
      +- Sort [match_id#2048 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(match_id#2048, 200), ENSURE_REQUIREMENTS, [plan_id=2385]
            +- BatchScan demo.bootcamp.matches_pb[match_id#2048, is_team_game#2049, playlist_id#2050, completion_date#2051] demo.bootcamp.matches_pb (branch=null) [filters=match_id

In [60]:
# the join keys are a subset of the partition/bucket
join_d = {'a':'matches_pb', 'b': 'matches_pb', 'on':['completion_date'], 'note__join_keys': 'subset'}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1784, match_id#1781, is_team_game#1782, playlist_id#1783, match_id#2074, is_team_game#2075, playlist_id#2076]
   +- SortMergeJoin [completion_date#1784], [completion_date#2077], Inner
      :- Sort [completion_date#1784 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(completion_date#1784, 200), ENSURE_REQUIREMENTS, [plan_id=2407]
      :     +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=null) [filters=completion_date IS NOT NULL, groupedBy=completion_date, match_id_bucket] RuntimeFilters: []
      +- Sort [completion_date#2077 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(completion_date#2077, 200), ENSURE_REQUIREMENTS, [plan_id=2408]
            +- BatchScan demo.bootcamp.matches_pb[match_id#2074, is_team_game#2075, playlist_id#2076, completion_date#2077] demo.bootcamp.

In [61]:
# matching both partitioning and bucking
join_d = {'a':'matches_pb', 'b': 'matches_pb', 'on':['completion_date', 'match_id']}
j = run_join_analysis(join_d)
j.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [completion_date#1784, match_id#1781, is_team_game#1782, playlist_id#1783, is_team_game#2101, playlist_id#2102]
   +- SortMergeJoin [completion_date#1784, match_id#1781], [completion_date#2103, match_id#2100], Inner
      :- Sort [completion_date#1784 ASC NULLS FIRST, match_id#1781 ASC NULLS FIRST], false, 0
      :  +- BatchScan demo.bootcamp.matches_pb[match_id#1781, is_team_game#1782, playlist_id#1783, completion_date#1784] demo.bootcamp.matches_pb (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=completion_date, match_id_bucket] RuntimeFilters: []
      +- Sort [completion_date#2103 ASC NULLS FIRST, match_id#2100 ASC NULLS FIRST], false, 0
         +- BatchScan demo.bootcamp.matches_pb[match_id#2100, is_team_game#2101, playlist_id#2102, completion_date#2103] demo.bootcamp.matches_pb (branch=null) [filters=completion_date IS NOT NULL, match_id IS NOT NULL, groupedBy=completion_date,

In [62]:
import pandas as pd

In [63]:
pd.DataFrame(join_d_l)

,a,b,on,has_shuffle,note__join_keys
0,matches_p,matches_p,[completion_date],False,NaN
1,matches_b,matches_b,[match_id],False,NaN
2,matches_pb,matches_pb,"[completion_date, match_id]",False,NaN
3,matches_p,matches_pd,[completion_date],False,NaN
4,matches_b,matches_bdm,[match_id],True,NaN
5,matches_b,matches_bd,[match_id],True,NaN
6,matches_p,matches_pb,[completion_date],True,NaN
7,matches_b,matches_pb,[match_id],True,NaN
8,matches_p,matches_p,"[completion_date, match_id]",False,superset
9,matches_b,matches_b,"[completion_date, match_id]",False,superset


In [ ]:
# again, the shuffle-free join with keys being a subset of the partition/buckets should be possible with spark>4.0
# by setting this config: 'spark.sql.sources.v2.bucketing.allowJoinKeysSubsetOfPartitionKeys.enabled'

In [ ]:
# scenarios 6 and 7 seem to work like special cases of a subset scenario,
# and as such, they do require shuffle.

In [ ]:
# the same goes to 4 and 5, they also seem to work like special cases of a subset scenario.
# probably, though, even with subset join working,
# case 5 (matches_b, matches_bd; diff numbers of buckets and not multiple) should require shuffle anyway

In [ ]:
# 'demo.bootcamp.matches_p' # partitioned
# 'demo.bootcamp.matches_b' # bucketed
# 'demo.bootcamp.matches_pb' # partitioned and bucketed
# 'demo.bootcamp.matches_pd' # with diff number of partitions
# 'demo.bootcamp.matches_bdm' # with diff number of buckets, but multiple
# 'demo.bootcamp.matches_bd' # with diff number of buckets

### Object storage (S3) connection

In [29]:
# stablishing connection with the minio,
# this way we can read the objct storage directly via cli,
# instead of using the CLI at http://localhost:9001/

!curl -L https://dl.min.io/aistor/mc/release/linux-amd64/mc -o /tmp/mc
!chmod +x /tmp/mc
!/tmp/mc alias set minio http://minio:9000 admin password
!/tmp/mc ls minio/

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  101M  100  101M    0     0  10.2M      0  0:00:09  0:00:09 --:--:-- 12.2M
]11;?\Added `minio` successfully.
]11;?\[2026-05-29 18:45:40 UTC]     0B warehouse/


In [73]:
!/tmp/mc ls minio/warehouse/bootcamp/

]11;?\[2026-05-29 23:01:58 UTC]     0B matches_b/
[2026-05-29 23:01:58 UTC]     0B matches_bd/
[2026-05-29 23:01:58 UTC]     0B matches_bdm/
[2026-05-29 23:01:58 UTC]     0B matches_p/
[2026-05-29 23:01:58 UTC]     0B matches_pb/
[2026-05-29 23:01:58 UTC]     0B matches_pd/


In [ ]:
# other option would be to access the object storage programatically,
# with boto3.

In [109]:
# !pip install boto3

In [32]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="admin",
    aws_secret_access_key="password",
    region_name="us-east-1",
)

def human(size):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.1f}{unit}"
        size /= 1024

def dir_size(prefix):
    total = 0
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(
        Bucket="warehouse",
        Prefix=prefix
    ):
        for obj in page.get("Contents", []):
            total += obj["Size"]
    return total

def ls(prefix=""):
    r = s3.list_objects_v2(
        Bucket="warehouse",
        Prefix=prefix,
        Delimiter="/"
    )

    # directories
    for p in r.get("CommonPrefixes", []):
        full_prefix = p["Prefix"]
        name = full_prefix[len(prefix):].rstrip("/")
        size = human(dir_size(full_prefix))
        print(f"{size:>8}  {name}/")

    # files
    for o in r.get("Contents", []):
        name = o["Key"][len(prefix):]
        if name and "/" not in name:
            size = human(o["Size"])
            print(f"{size:>8}  {name}")

In [33]:
resp = s3.list_objects_v2(Bucket="warehouse")

In [76]:
ls("bootcamp/matches_pb/data/completion_date=2016-01-01/")

   3.5KB  match_id_bucket=1/
   3.6KB  match_id_bucket=10/
   3.4KB  match_id_bucket=11/
   3.4KB  match_id_bucket=12/
   3.6KB  match_id_bucket=13/
   3.5KB  match_id_bucket=14/
   3.4KB  match_id_bucket=15/
   3.6KB  match_id_bucket=2/
   3.3KB  match_id_bucket=3/
   3.5KB  match_id_bucket=4/
   3.5KB  match_id_bucket=5/
   3.5KB  match_id_bucket=6/
   3.5KB  match_id_bucket=7/
   3.5KB  match_id_bucket=8/
   3.7KB  match_id_bucket=9/


In [216]:
!/tmp/mc ls minio/warehouse/bootcamp/

]11;?\

In [191]:
# !/tmp/mc rm -r --force minio/warehouse/bootcamp/match_details_bucketed

### DROP TABLE PURGE
Clean env

In [77]:
t_types = [
    'demo.bootcamp.matches_p', # partitioned
    'demo.bootcamp.matches_b', # bucketed
    'demo.bootcamp.matches_pb', # partitioned and bucketed
    'demo.bootcamp.matches_pd', # with diff number of partitions
    'demo.bootcamp.matches_bdm', # with diff number of buckets, but multiple
    'demo.bootcamp.matches_bd' # with diff number of buckets
]

df_d = {}
for t in t_types:
    spark.sql(f"drop table if exists {t} purge")
    print(f'drop table if exists {t} purge;')

drop table if exists demo.bootcamp.matches_p purge;


drop table if exists demo.bootcamp.matches_b purge;


drop table if exists demo.bootcamp.matches_pb purge;


drop table if exists demo.bootcamp.matches_pd purge;


drop table if exists demo.bootcamp.matches_bdm purge;


[Stage 237:===================================================> (199 + 5) / 204]

drop table if exists demo.bootcamp.matches_bd purge;
